# LELA60331 Week 7 Seminar: Finishing logistic regression

In Week 5 we built logistic regression one step at a time, and got as far as making **one** improvement to the weights (Part 6). This week we finish the job on the simulated data: **training**, the **learning rate**, **using** the model and **evaluating** it. Then we meet a new tool, **matrix-vector multiplication**, which does the sums for every text and every feature at once.

| Part | What you do |
|---|---|
| 0 | Run the set-up cell: everything you built in Week 5, Parts 1–6 |
| 1 | Repeat the improvement step many times: **training** |
| 2 | Experiment with the learning rate |
| 3 | Use the trained model |
| 4 | Evaluate it with precision and recall |
| 5 | Matrix-vector multiplication: doing the sums for every text at once |

As in Week 5, each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell. Run the check: ✅ means your answer is right, ❌ means not yet. Many activities also tell you what output you should get.

We use **plain Python** (lists, loops, functions and list comprehensions) plus `math`, `random` and `matplotlib`.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Where we got to in Week 5

The cell below sets up everything you built in Week 5, Parts 1–6, so that this sheet works on its own. Read through it to remind yourself of each piece, then run it.

| Name | What it does | Week 5 |
|---|---|---|
| `dot(a, b)` | the dot product of two lists | Week 4 |
| `score(x1, x2, weights, bias)` | `z = bias + x1 * weight1 + x2 * weight2` | Activity 1 |
| `sigmoid(z)` | turns a score into a probability between 0 and 1 | Activity 2 |
| `classify(p)` | 1 if `p` is above 0.5, otherwise 0 | Activity 3 |
| `x`, `y` | the 50 simulated texts: `x[0]` and `x[1]` hold the two features, `y` the labels | Part 3 |
| `loss_one(q, label)`, `average_loss(q, y)` | the cross-entropy loss | Activities 6 and 7 |
| `plot_data()`, `plot_decision_line(weights, bias)` | draw the texts and the decision boundary | Parts 3 and 4 |

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length (from Week 4)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def score(x1, x2, weights, bias):
    return bias + x1 * weights[0] + x2 * weights[1]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def classify(p):
    if p > 0.5:
        return 1
    else:
        return 0

# the simulated data from Week 5, Part 3
random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
num_samples = len(y)

def loss_one(q, label):
    if label == 1:
        return -math.log2(q)
    else:
        return -math.log2(1 - q)

def average_loss(q, y):
    losses = [loss_one(q[k], y[k]) for k in range(len(y))]
    return sum(losses) / len(losses)

def plot_data():
    for k in range(len(y)):
        if y[k] == 1:
            plt.scatter(x[0][k], x[1][k], marker="o", color="tab:orange", s=60)
        else:
            plt.scatter(x[0][k], x[1][k], marker="*", color="tab:blue", s=100)
    plt.xlabel("x1: log count of positive words")
    plt.ylabel("x2: log count of negative words")
    plt.xlim(0, 5)
    plt.ylim(0, 5)

def plot_decision_line(weights, bias):
    plot_data()
    # the line where z = 0, i.e. where the probability is exactly 0.5
    line_x = [0, 5]
    line_y = [-(bias + weights[0] * value) / weights[1] for value in line_x]
    plt.plot(line_x, line_y, "k--")
    plt.show()

print("Ready:", num_samples, "texts,", sum(y), "of them positive")

In Week 5, Part 6 we made **one** improvement to an untrained model:

1. calculate the probability `q` of every text being positive
2. calculate the errors, `q - y`, and from them the gradients `dw1`, `dw2` and `db`
3. move each weight and the bias a little way against its gradient: **new value = old value − learning rate × gradient**

That one step made the loss go down from 2.065 to 1.624. Training simply repeats the step many times.

---
# Part 1 · Training: repeating the step many times

### Activity 1 — The training loop

Everything inside the loop is something you wrote in Week 5, Activities 5–9. Each pass through the loop:

1. calculates the probabilities `q` with the current weights and bias
2. records the loss
3. calculates the errors and the gradients
4. updates the weights and the bias

Fill in the blanks. (One pass through the training data is called an **epoch**.)

In [ ]:
random.seed(10)
weights = [random.random(), random.random()]
bias = 0
lr = 0.5
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):
    # 1. predictions
    q = ???
    # 2. loss
    loss_history.append(???)
    # 3. errors and gradients
    errors = ???
    dw1 = ???
    dw2 = ???
    db = ???
    # 4. update
    weights = ???
    bias = ???

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print("weights:", weights, " bias:", bias)     # about [1.17, -1.32] and 0.51

In [ ]:
# ✅ Check
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("final loss about 0.656", lambda: abs(loss_history[-1] - 0.656) < 0.001)
check("weight for positive words is positive", lambda: weights[0] > 0)
check("weight for negative words is negative", lambda: weights[1] < 0)

The loss falls quickly at first and then levels off. The weights now make sense: positive words push the score up and negative words push it down. Let's look at the new decision boundary.

In [ ]:
plot_decision_line(weights, bias)

### Activity 2 — Accuracy after training

Calculate the accuracy of the trained model, as in Week 5, Activity 5. Before training it was 0.46. How much has it improved? Look at the plot: which texts does it still get wrong, and why can no straight line get them right?

In [ ]:
q = ???
predictions = ???
accuracy = ???
print("accuracy:", accuracy)       # you should get 0.78

In [ ]:
# ✅ Check
check("accuracy", lambda: accuracy == 0.78)

---
# Part 2 · The learning rate

### Activity 3 — Experiment

Go back to Activity 1 and change `lr`, then run the training loop and the cells below it again.

1. Try `lr = 0.01`. What happens to the loss curve? Is 1000 epochs enough?
2. Try `lr = 2`. What happens to the loss curve now? Why?
3. Set `lr` back to 0.5 and run the training loop again before you carry on.

Compare what you see with Activities 2 and 3 of the Week 4 sheet.

*Write your observations here.*

---
# Part 3 · Using the trained model

### Activity 4 — Classifying new texts

Complete `p_positive(n_positive, n_negative)`, which returns the probability that a text with the given **counts** of positive and negative words is positive. Remember that the model's features are the **logs** of the counts, so use `math.log` on each count first.

What is the probability that each of these texts is positive?

a) 2 positive words and 3 negative words
b) 10 positive words and 1 negative word

In [ ]:
def p_positive(n_positive, n_negative):
    x1 = ???
    x2 = ???
    return ???

print(round(p_positive(2, 3), 3))      # you should get 0.466
print(round(p_positive(10, 1), 3))     # you should get 0.961

In [ ]:
# ✅ Check
check("2 positive, 3 negative", lambda: abs(p_positive(2, 3) - 0.466) < 0.005)
check("10 positive, 1 negative", lambda: abs(p_positive(10, 1) - 0.961) < 0.005)

⭐ **Challenge (optional):** what happens if you call `p_positive(0, 3)`? Why? How could you change the features to avoid this problem?

---
# Part 4 · Evaluation: precision and recall

Accuracy can be misleading. If 90% of texts were negative, a model that always said "negative" would have an accuracy of 0.9, but it would never find a single positive text. So we also measure **precision** and **recall** for the positive class:

* a **true positive** is a positive text that the model predicted as positive
* a **false positive** is a negative text that the model wrongly predicted as positive
* a **false negative** is a positive text that the model wrongly predicted as negative

**Precision** = true positives / (true positives + false positives): of the texts the model *called* positive, how many really were?

**Recall** = true positives / (true positives + false negatives): of the texts that really *were* positive, how many did the model find?

### Activity 5 — Precision and recall

Complete the counting loop and the two calculations, using `predictions` from Activity 2. The condition for true positives is done for you.

In [ ]:
tp = 0
fp = 0
fn = 0
for k in range(num_samples):
    if predictions[k] == 1 and y[k] == 1:
        tp += 1
    if ???:          # predicted positive, but really negative
        fp += 1
    if ???:          # predicted negative, but really positive
        fn += 1

precision = ???
recall = ???
print(tp, fp, fn)                  # you should get 18 6 5
print("precision:", round(precision, 3), " recall:", round(recall, 3))

In [ ]:
# ✅ Check
check("the counts add up", lambda: tp + fn == sum(y))
check("precision", lambda: precision == tp / (tp + fp))
check("recall", lambda: recall == tp / (tp + fn))

Which is higher, precision or recall? What does that tell you about the mistakes this model makes?

That completes logistic regression on the simulated data.

---
# Part 5 · Matrix-vector multiplication

So far each text has had just **two** features, and we have written a separate piece of code for each one: `x1 * weight1 + x2 * weight2`, and separate gradients `dw1` and `dw2`. Real text needs far more features, for example one for every word in a vocabulary of thousands of words. We can't write thousands of lines of code, so we need a way of doing the sums for **all** the features, and **all** the texts, at once. That way is **matrix-vector multiplication**.

We'll build up to it in small steps:

1. what a matrix is, and how to get things out of one
2. our data as a matrix
3. the score of **one** text, using a row of the matrix
4. the scores of **all** the texts at once: matrix-vector multiplication
5. the gradients, using the same trick

## Step 1 · Matrices

A **matrix** is a table of numbers with rows and columns. In plain Python we store a matrix as a **list of lists**, with one inner list for each row (Week 4 Python workbook). Here is a matrix with 2 rows and 3 columns:

$$m = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}$$

In Python:

```python
m = [[1, 2, 3],      # row 0
     [4, 5, 6]]      # row 1
```

* `m[0]` is row 0: the list `[1, 2, 3]`
* `m[0][2]` is row 0, column 2: the number `3`. Read it in two steps: `m[0]` gets the row, then `[2]` gets the element from that row.
* `len(m)` is the number of **rows** (the number of inner lists), and `len(m[0])` is the number of **columns** (the length of one row)

Run the cell to see these.

In [ ]:
m = [[1, 2, 3],
     [4, 5, 6]]

print(m[0])            # row 0
print(m[0][2])         # row 0, column 2
print(len(m))          # the number of rows
print(len(m[0]))       # the number of columns
for row in m:          # a for loop over a matrix gives one row at a time
    print("a row:", row)

### Activity 6 — Getting things out of a matrix

Using `m`, set:

1. `row_1`: row 1 of `m`
2. `element_1_2`: the element in row 1, column 2
3. `element_0_1`: the element in row 0, column 1
4. `n_rows` and `n_columns`: the number of rows and the number of columns of `m`

In [ ]:
row_1 = ???
element_1_2 = ???
element_0_1 = ???
n_rows = ???
n_columns = ???
print(row_1, element_1_2, element_0_1, n_rows, n_columns)     # you should get [4, 5, 6] 6 2 2 3

In [ ]:
# ✅ Check
check("row_1", lambda: row_1 == [4, 5, 6])
check("element_1_2", lambda: element_1_2 == 6)
check("element_0_1", lambda: element_0_1 == 2)
check("n_rows and n_columns", lambda: n_rows == 2 and n_columns == 3)

## Step 2 · Our data as a matrix

Our data is stored as `x`, a list of two lists: `x[0]` holds feature 1 for **every** text, and `x[1]` holds feature 2 for every text. Run the cell to see the start of each.

In [ ]:
print(x[0][:3])        # feature 1 for texts 0, 1 and 2
print(x[1][:3])        # feature 2 for texts 0, 1 and 2
print(x[0][0], x[1][0])    # the two features of text 0

For matrix-vector multiplication we want the data the other way round: a matrix `X` (capital X) with **one row per text** and **one column per feature**. Text `k`'s row is `[x[0][k], x[1][k]]`, so:

| | column 0 (feature 1) | column 1 (feature 2) |
|---|---|---|
| row 0 (text 0) | `x[0][0]` | `x[1][0]` |
| row 1 (text 1) | `x[0][1]` | `x[1][1]` |
| ... | ... | ... |
| row 49 (text 49) | `x[0][49]` | `x[1][49]` |

### Activity 7 — Building `X` with a loop

Complete the loop. For each text `k`, it should make the list `row` with that text's two features, then add `row` to the end of `X`.

In [ ]:
X = []
for k in range(num_samples):
    row = [???, ???]       # text k's two features
    ???                   # add the row to the end of X

print(len(X), len(X[0]))      # you should get 50 2: 50 rows, 2 columns
print(X[0])                   # text 0's two features

In [ ]:
# ✅ Check
check("50 rows", lambda: len(X) == 50)
check("2 columns in every row", lambda: all(len(row) == 2 for row in X))
check("row 0", lambda: X[0] == [x[0][0], x[1][0]])
check("row 49", lambda: X[49] == [x[0][49], x[1][49]])

### The same thing as a list comprehension

A loop that builds a list by appending one item each time can be written more compactly as a **list comprehension** (Week 4 Python workbook). The comprehension contains the same two pieces as the loop, in a different order: **what to add**, then **the `for` part**.

| Loop | List comprehension |
|---|---|
| `squares = []`<br>`for k in range(5):`<br>`    squares.append(k * k)` | `squares = [k * k for k in range(5)]` |
| `X = []`<br>`for k in range(num_samples):`<br>`    X.append([x[0][k], x[1][k]])` | `X = [??? for k in range(num_samples)]` |

Read `[k * k for k in range(5)]` as "a list of `k * k`, for each `k` from 0 to 4". Run the cell to check that the two versions agree.

In [ ]:
squares = []
for k in range(5):
    squares.append(k * k)
print(squares)
print([k * k for k in range(5)])

### Activity 8 — Building `X` with a list comprehension

Fill in the gap in the bottom row of the table: write `X2`, the same matrix as `X`, as a list comprehension. *Hint:* the "what to add" part is the row for text `k`, which is itself a list.

In [ ]:
X2 = [??? for k in range(num_samples)]
print(X2 == X)          # you should get True

In [ ]:
# ✅ Check
check("X2 is the same as X", lambda: X2 == X)

## Step 3 · The score of one text

In Week 4 you wrote `dot(a, b)`, the **dot product** of two lists: multiply them element by element, then add up. For text 0, the dot product of its row `X[0]` with `weights` is

`X[0][0] * weights[0] + X[0][1] * weights[1]`

which is `x1 * weight1 + x2 * weight2`: exactly the score of text 0 without the bias. Add the bias, and you have the score.

### Activity 9 — One row at a time

Using the trained `weights` and `bias` from Part 1:

1. `score_0`: the score of text 0, using `dot` with row 0 of `X`, plus the bias
2. `score_1`: the same for text 1

In [ ]:
score_0 = ???
score_1 = ???
print(score_0, score(x[0][0], x[1][0], weights, bias))     # the two numbers should be the same
print(score_1, score(x[0][1], x[1][1], weights, bias))

In [ ]:
# ✅ Check
check("score_0", lambda: abs(score_0 - score(x[0][0], x[1][0], weights, bias)) < 1e-9)
check("score_1", lambda: abs(score_1 - score(x[0][1], x[1][1], weights, bias)) < 1e-9)

## Step 4 · All the scores at once: matrix-vector multiplication

Doing that for **every** row is called **matrix-vector multiplication**: take the dot product of **each row** of the matrix with the vector. You get one number per row.

Here is a small example worked through. Multiply the matrix `m` from Step 1 by the vector `[1, 0, 2]`:

* row 0 is `[1, 2, 3]`: its dot product with `[1, 0, 2]` is 1 × 1 + 2 × 0 + 3 × 2 = 1 + 0 + 6 = **7**
* row 1 is `[4, 5, 6]`: its dot product with `[1, 0, 2]` is 4 × 1 + 5 × 0 + 6 × 2 = 4 + 0 + 12 = **16**

So the answer is `[7, 16]`. In maths notation:

$$\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix} \begin{bmatrix} 1 \\ 0 \\ 2 \end{bmatrix} = \begin{bmatrix} 1 \times 1 + 2 \times 0 + 3 \times 2 \\ 4 \times 1 + 5 \times 0 + 6 \times 2 \end{bmatrix} = \begin{bmatrix} 7 \\ 16 \end{bmatrix}$$

Notice the shapes: the matrix has **3 columns** and the vector has **3 elements** (they must match), and the answer has one number for each of the **2 rows**.

### Activity 10 — Matrix-vector multiplication by hand

Work out, on paper, the matrix `m` multiplied by the vector `[2, 1, 1]`. Then write your answer as a list of two numbers.

In [ ]:
by_hand = [???, ???]

In [ ]:
# ✅ Check
check("row 0: 1 × 2 + 2 × 1 + 3 × 1", lambda: by_hand[0] == 7)
check("row 1: 4 × 2 + 5 × 1 + 6 × 1", lambda: by_hand[1] == 19)

### Activity 11 — A matrix-vector multiplication function

Now let the computer do it. Complete `mat_vec(matrix, vector)`. It goes through the matrix one row at a time (as in the `for row in m` loop in Step 1) and adds the dot product of that row with `vector` to the list `result`.

In [ ]:
def mat_vec(matrix, vector):
    result = []
    for row in matrix:
        result.append(???)      # the dot product of this row with the vector
    return result

print(mat_vec(m, [1, 0, 2]))     # you should get [7, 16]
print(mat_vec(m, [2, 1, 1]))     # your answer from Activity 10

In [ ]:
# ✅ Check
check("mat_vec(m, [1, 0, 2])", lambda: mat_vec(m, [1, 0, 2]) == [7, 16])
check("mat_vec(m, [2, 1, 1])", lambda: mat_vec(m, [2, 1, 1]) == [7, 19])
check("a 3 × 2 matrix", lambda: mat_vec([[1, 0], [0, 1], [1, 1]], [5, 3]) == [5, 3, 8])

### Activity 12 — The score of every text

Now use `mat_vec` on our data. `mat_vec(X, weights)` gives `x1 * weight1 + x2 * weight2` for every text: one number per row of `X`.

1. `scores_without_bias`: `mat_vec` with `X` and the trained `weights`
2. `z_all`: the full score of every text. Complete the loop so that it adds the bias to each number in `scores_without_bias`.

In [ ]:
scores_without_bias = ???

z_all = []
for value in scores_without_bias:
    z_all.append(???)

print(len(z_all))                 # you should get 50: one score per text
print(z_all[0], score_0)          # the same as in Activity 9

In [ ]:
# ✅ Check
check("scores_without_bias", lambda: len(scores_without_bias) == 50 and abs(scores_without_bias[0] - dot(X[0], weights)) < 1e-9)
check("z_all has one score per text", lambda: len(z_all) == 50)
check("z_all gives the same scores as score()",
      lambda: all(abs(z_all[k] - score(x[0][k], x[1][k], weights, bias)) < 1e-9 for k in range(50)))

⭐ **Challenge (optional):** write `z_all` as a list comprehension in a single line. *Hint:* "a list of `value + bias`, for each `value` in `mat_vec(X, weights)`".

### Shapes matter

Each row of `X` has 2 elements, so `X` can only be multiplied by a vector with 2 elements. Here is a version of `dot` that checks this and gives an error if the lengths don't match.

🔮 **Predict:** the cell below tries `mat_vec(X, [1, 2])` and `mat_vec(X, [1, 2, 3])`. Which one will fail, and why? Then run it.

In [ ]:
def dot(a, b):
    # the dot product of two lists of the same length
    if len(a) != len(b):
        raise ValueError("the two lists must have the same length")
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

print(mat_vec(X, [1, 2])[:3])          # fine: 2 columns, 2 weights
try:
    mat_vec(X, [1, 2, 3])              # 2 columns, but 3 weights
except ValueError as err:
    print("ValueError:", err)

(`try` and `except` catch the error, so that the rest of the cell, and the rest of the notebook, keeps running.)

## Step 5 · The gradients are a matrix-vector multiplication too

In Week 5 we calculated the gradients like this:

* `dw1 = dot(x[0], errors) / num_samples`
* `dw2 = dot(x[1], errors) / num_samples`

Look at what `x[0]` and `x[1]` are: `x[0]` is feature 1 for every text, which is **column 0** of `X`, and `x[1]` is **column 1** of `X`. So `x` is `X` with its rows turned into columns. A matrix with its rows turned into columns is called the **transpose**.

| `X`: one row per text | `x`: one row per feature (the transpose of `X`) |
|---|---|
| `X[0]` = `[x1 of text 0, x2 of text 0]` | `x[0]` = `[x1 of text 0, x1 of text 1, ...]` |
| `X[1]` = `[x1 of text 1, x2 of text 1]` | `x[1]` = `[x2 of text 0, x2 of text 1, ...]` |
| ... (50 rows) | (2 rows) |

### Activity 13 — A column of `X` is a row of `x`

Complete the loop so that `column_0` collects element 0 of every row of `X`. Is it the same as `x[0]`?

In [ ]:
column_0 = []
for row in X:
    column_0.append(???)

print(column_0[:3])
print(x[0][:3])
print(column_0 == x[0])          # you should get True

In [ ]:
# ✅ Check
check("column_0 is x[0]", lambda: column_0 == x[0])

So each gradient is the dot product of **one row of `x`** with the errors. Doing that for every row of `x` is, again, a matrix-vector multiplication: `mat_vec(x, errors)` gives `[dot(x[0], errors), dot(x[1], errors)]`, one number per feature.

### Activity 14 — All the gradients at once

Using the trained model's probabilities `q` from Activity 2:

1. `errors`: complete the loop so that it adds `q[k] - y[k]` for every text (as in Week 5)
2. `gradient_sums`: one call to `mat_vec`, with `x` and `errors`
3. `gradients`: complete the loop so that it divides each number in `gradient_sums` by `num_samples`

In [ ]:
errors = []
for k in range(num_samples):
    errors.append(???)                 # 1.

gradient_sums = ???             # 2.

gradients = []
for value in gradient_sums:
    gradients.append(???)      # 3.

print(len(gradients))         # you should get 2: one gradient per weight
print(gradients)              # both are close to 0: the trained model is near the bottom of the loss

In [ ]:
# ✅ Check
check("1. errors", lambda: len(errors) == 50 and abs(errors[0] - (q[0] - y[0])) < 1e-12)
check("2. gradient_sums", lambda: len(gradient_sums) == 2 and abs(gradient_sums[0] - dot(x[0], errors)) < 1e-9)
check("3. gradient for weight 1", lambda: abs(gradients[0] - dot(x[0], errors) / num_samples) < 1e-9)
check("3. gradient for weight 2", lambda: abs(gradients[1] - dot(x[1], errors) / num_samples) < 1e-9)

So training needs just two matrix-vector multiplications, whatever the number of features:

* **matrix (one row per text) × weights** → a score for every text
* **transpose (one row per feature) × errors** → a gradient for every weight



---
## ✏️ Summary

In this sheet you have:

1. **trained** logistic regression by repeating the improvement step many times
2. seen how the **learning rate** affects training
3. **used** the trained model to classify new texts
4. **evaluated** it with accuracy, precision and recall
5. used **matrix-vector multiplication** to calculate the scores of every text, and the gradients of every weight, at once